# 08 Evaluation & Safety：评估、幻觉与安全

> 前置：本模块全部课程。
> 目标：建立"怎么证明模型真的变好了"的评估体系；理解幻觉、越狱与对齐失效；收束 LLM 模块全景。

## 1. 评估的三个层次

| 层次 | 方法 | 例子 |
|---|---|---|
| 预训练质量 | PPL / 交叉熵（无需标注） | 本课演示 |
| 能力基准 | 考试式评测集 | MMLU、GSM8K、HumanEval |
| 对齐质量 | 人类偏好 / 红队 | Helpful&Harmless、TruthfulQA |

**PPL 与能力的错位**：PPL 只测"概率上像不像人话"，不测"对不对"。两个模型 PPL 相同，一个可能满口胡言——所以必须上基准。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


In [ ]:
class CharTokenizer:
    """字符级分词器：LLM 的最小原型。"""
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.itos = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s):
        return [self.stoi[c] for c in s]
    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)

class MHA(nn.Module):
    def __init__(self, d=32, n_heads=4):
        super().__init__()
        assert d % n_heads == 0
        self.dk = d // n_heads; self.n_heads = n_heads
        self.Wq = nn.Linear(d, d); self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d); self.Wo = nn.Linear(d, d)
        self.register_buffer("mask", torch.triu(torch.ones(512, 512), diagonal=1).bool())
    def forward(self, x):
        B, T, d = x.shape
        q = self.Wq(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.n_heads, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        scores = scores.masked_fill(self.mask[:T, :T], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        return self.Wo((attn @ v).transpose(1, 2).contiguous().view(B, T, d))

class Block(nn.Module):
    def __init__(self, d=32, n_heads=4, d_ff=64):
        super().__init__()
        self.norm1 = nn.LayerNorm(d); self.attn = MHA(d, n_heads)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ff(self.norm2(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d=32, n_heads=4, d_ff=64, n_layers=2, max_t=128):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(max_t, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos[:T]
        for b in self.blocks:
            x = b(x)
        return self.head(self.norm(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

def make_corpus(seed=0, length=40000, style="default"):
    """生成有结构的合成语料（字符级）。style='reversed' 时词序倒装。"""
    rng = np.random.default_rng(seed)
    subjects = ["cat", "dog", "bird", "fox"]
    verbs = ["sat", "ran", "sang", "jumped"]
    places = ["mat", "park", "tree", "fence"]
    text = ""
    while len(text) < length:
        s = "the " + rng.choice(subjects) + " " + rng.choice(verbs) + " on the " + rng.choice(places)
        if style == "reversed":
            s = " ".join(reversed(s.split()))
        text += s + " "
    return text[:length]

def train_gpt(model, ids, steps=500, batch=64, seq=32, lr=3e-3, val_ids=None, log_every=100):
    """标准训练循环，返回 (steps, train_loss) 列表。"""
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    n = len(ids) - seq - 1
    curve = []
    for step in range(steps):
        i = torch.randint(0, n, (batch,))
        x = torch.stack([ids[j:j + seq] for j in i])
        y = torch.stack([ids[j + 1:j + seq + 1] for j in i])
        opt.zero_grad()
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        loss.backward(); opt.step()
        if step % log_every == 0 or step == steps - 1:
            curve.append((step, loss.item()))
            if val_ids is not None and step % (log_every * 5) == 0:
                with torch.no_grad():
                    vi = torch.randint(0, len(val_ids) - seq - 1, (64,))
                    vx = torch.stack([val_ids[j:j + seq] for j in vi])
                    vy = torch.stack([val_ids[j + 1:j + seq + 1] for j in vi])
                    vl = F.cross_entropy(model(vx).reshape(-1, model.head.out_features), vy.reshape(-1)).item()
                print(f"step {step:4d}  train={loss.item():.4f}  val={vl:.4f}")
            else:
                print(f"step {step:4d}  train={loss.item():.4f}")
    return curve

@torch.no_grad()
def generate(model, tok, prompt, n=80, temp=0.8):
    ids = tok.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long)
    for _ in range(n):
        logits = model(idx[:, -64:])
        probs = torch.softmax(logits[:, -1, :] / temp, dim=-1)
        nxt = torch.multinomial(probs, 1)
        idx = torch.cat([idx, nxt], dim=1)
    return tok.decode(idx[0].tolist())


## PPL 实测

训练一个小模型，在**未见过的验证语料**上算 PPL。PPL 是预训练质量的第一道门。

In [ ]:
corpus = make_corpus(seed=0, length=30000)
tok = CharTokenizer(corpus)
ids = torch.tensor(tok.encode(corpus), dtype=torch.long)
split = int(len(ids) * 0.9)
train_ids, val_ids = ids[:split], ids[split:]

torch.manual_seed(0)
model = TinyGPT(tok.vocab_size, d=32, n_heads=4, d_ff=64, n_layers=2, max_t=64)
train_gpt(model, train_ids, steps=600, seq=32, log_every=300)

with torch.no_grad():
    vi = torch.randint(0, len(val_ids) - 33, (200,))
    vx = torch.stack([val_ids[j:j + 32] for j in vi])
    vy = torch.stack([val_ids[j + 1:j + 33] for j in vi])
    val_loss = F.cross_entropy(model(vx).reshape(-1, tok.vocab_size), vy.reshape(-1)).item()
ppl = math.exp(val_loss)
print(f"验证 loss = {val_loss:.4f}  →  PPL = {ppl:.3f}")

## 2. 能力基准（真实 LLM 的"高考"）

| 基准 | 测什么 | 典型满分口径 |
|---|---|---|
| MMLU | 57 学科知识问答 | ~90% (GPT-4o) |
| GSM8K | 小学数学应用题 | ~95% |
| HumanEval | 代码生成 | ~90% pass@1 |
| HellaSwag | 常识推理 | ~95% |
| TruthfulQA | 事实性与幻觉 | 偏低是常态 |

> 教训：单一基准可刷；真正部署要组合评估 + 人工抽检。评测集污染（模型训练数据里见过题）是 2023 年后的公认风险。

## 3. 幻觉：为什么模型会编造

模型学的是"下一个 token 的概率分布"，不是"事实数据库"。在知识边界处，最高概率的续写往往是**流畅但不存在的说法**。缓解手段：

1. RAG（07 课）：给模型材料，减少"自由发挥"；
2. 解码约束：低温度、采样后验校验；
3. 训练时：SFT 教"不知道就说不知道"，RLHF/DPO 用 TruthfulQA 类偏好对打压幻觉；
4. 部署时：引用来源 + 置信度阈值。

> 幻觉**无法完全消除**（模型本质是条件概率生成器），只能压制——这是所有 LLM 产品的工程红线。

## 4. 安全与对齐失效模式

| 威胁 | 例子 | 对策 |
|---|---|---|
| 越狱（jailbreak） | "假装是开发者模式…" | 红队测试 + 输入过滤 + 持续 SFT |
| 提示注入 | 网页/文档里藏指令劫持 Agent | 权限隔离 + 指令与数据分离 |
| 数据投毒 | 训练语料混入恶意样本 | 数据审查 + 水印 |
| 对齐漂移 | 模型越来越顺从到有害 | 对齐评估持续监控 |
| 偏见放大 | 训练数据的社会偏见 | 公平性评估 + 微调去偏 |

**工程现实**：安全不是一次训练，而是**评估-发现-修补**的持续循环（red teaming）。

## 5. LLM 模块全景回顾

```
01 分词(BPE) → 02 规模定律 → 03 预训练 → 04 微调/LoRA
                                          ↓
                      05 对齐(RLHF/DPO) → 06 推理优化(KV/量化)
                                          ↓
                      07 应用(RAG/Agent) ← 08 评估与安全(护栏)
```

**贯穿主线**：
- 所有环节都在**预测下一个 token**（01-03）；
- 所有改进都在**约束这个预测**（04-06 约束到任务/资源，07-08 约束到事实与安全）。

## 课后练习

1. 在同一个模型上分别用温度 0.1 / 1.0 / 3.0 生成 10 句，统计"重复度"——直观理解解码参数如何影响幻觉与多样性。
2. 给 TinyGPT 喂"fact" 语料（`"paris is the capital of france "` ×100），再喂"谎言"语料微调 100 步，观察它如何学会编造——亲手制造一次"幻觉"。
3. 调研一个你关注的模型的公开评估卡（model card），列出它的能力边界与安全声明。